# Chandler Wobble Analysis from IERS Data

## Overview

The **Chandler wobble** is a small periodic motion of Earth's rotational axis relative to the Earth's surface. 
Key characteristics:
- **Period**: ~433 days (~14.2 months) - this differs from the theoretical 305-day Euler period due to Earth's non-rigidity
- **Amplitude**: ~0.1-0.2 arcseconds (variable over time)
- **Discovery**: Seth Carlo Chandler, 1891

## Data Source

We use the IERS EOP C04 data series which contains:
- **x, y**: Polar motion coordinates (arcseconds)
- Daily resolution from 1962 to present

## Confounding Factors to Isolate

1. **Annual wobble** (~365.25 days): Largest signal, caused by seasonal redistribution of atmospheric and oceanic mass
2. **Semi-annual wobble** (~182.6 days): Secondary seasonal signal
3. **Long-term polar drift**: Secular motion of the pole (~10 mas/year toward ~80°W longitude)
4. **Decadal variations**: Long-term amplitude modulations
5. **Geophysical excitations**: Atmospheric, oceanic, and hydrological angular momentum
6. **Measurement noise**: Instrumental and processing errors

## Analysis Approach

1. Visualize raw polar motion time series
2. Spectral analysis to identify frequency components
3. Remove linear trend (polar drift)
4. Design band-pass filter for Chandler frequency band
5. Extract isolated Chandler wobble signal
6. Analyze amplitude variations over time

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import signal
from scipy.fft import fft, fftfreq
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# Project imports
import sys
sys.path.insert(0, '../src')
from ecdo_analysis import load_polar_motion_data

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# For saving GitHub-compatible artifacts
FIGURES_DIR = '../figures'
import os
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Setup complete!")

## 1. Load Polar Motion Data

In [ ]:
# Load daily polar motion data
df = load_polar_motion_data('../data/EOP_14_C04_IAU2000A_one_file_1962-now.txt')

print(f"Data range: {df.index.min()} to {df.index.max()}")
print(f"Number of observations: {len(df)}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Basic statistics
print("Polar Motion Statistics (arcseconds):")
print("="*50)
df[['x', 'y']].describe()

## 2. Visualize Raw Polar Motion

The polar motion is typically represented as:
- **x**: Component along 0° longitude (Greenwich meridian)
- **y**: Component along 90°W longitude

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# X component
axes[0].plot(df.index, df['x'], 'b-', linewidth=0.5, alpha=0.7)
axes[0].set_ylabel('x (arcsec)')
axes[0].set_title('Polar Motion X Component (toward Greenwich meridian)')
axes[0].grid(True, alpha=0.3)

# Y component
axes[1].plot(df.index, df['y'], 'r-', linewidth=0.5, alpha=0.7)
axes[1].set_ylabel('y (arcsec)')
axes[1].set_title('Polar Motion Y Component (toward 90°W)')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/polar_motion_raw_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR}/polar_motion_raw_timeseries.png")

In [ ]:
# Plot polar motion in the x-y plane (shows the spiral path of the pole)
fig, ax = plt.subplots(figsize=(10, 10))

# Color by time
colors = np.linspace(0, 1, len(df))
scatter = ax.scatter(df['x'], df['y'], c=colors, cmap='viridis', s=0.5, alpha=0.5)

# Mark start and end
ax.scatter(df['x'].iloc[0], df['y'].iloc[0], c='green', s=100, marker='o', label=f'Start ({df.index[0].year})', zorder=5)
ax.scatter(df['x'].iloc[-1], df['y'].iloc[-1], c='red', s=100, marker='s', label=f'End ({df.index[-1].year})', zorder=5)

ax.set_xlabel('x (arcsec) - toward Greenwich')
ax.set_ylabel('y (arcsec) - toward 90°W')
ax.set_title('Polar Motion Trajectory (x-y plane)\nShows secular drift + wobble')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax, label='Time progression')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/polar_motion_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Remove Linear Trend (Secular Polar Drift)

The pole drifts slowly over time. We need to remove this trend to isolate oscillatory signals.

In [ ]:
# Calculate days since start for trend fitting
t_days = (df.index - df.index[0]).total_seconds() / 86400

# Fit linear trend to x and y
x_trend_coeffs = np.polyfit(t_days, df['x'], 1)
y_trend_coeffs = np.polyfit(t_days, df['y'], 1)

x_trend = np.polyval(x_trend_coeffs, t_days)
y_trend = np.polyval(y_trend_coeffs, t_days)

# Calculate trend rates in mas/year
x_rate_mas_yr = x_trend_coeffs[0] * 365.25 * 1000  # Convert arcsec/day to mas/year
y_rate_mas_yr = y_trend_coeffs[0] * 365.25 * 1000

print(f"Polar drift rates:")
print(f"  x: {x_rate_mas_yr:.2f} mas/year")
print(f"  y: {y_rate_mas_yr:.2f} mas/year")
print(f"  Total drift: {np.sqrt(x_rate_mas_yr**2 + y_rate_mas_yr**2):.2f} mas/year")
print(f"  Direction: {np.degrees(np.arctan2(y_rate_mas_yr, x_rate_mas_yr)):.1f}°")

In [ ]:
# Detrend the data
df['x_detrend'] = df['x'] - x_trend
df['y_detrend'] = df['y'] - y_trend

# Plot detrended data
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df.index, df['x_detrend'], 'b-', linewidth=0.5, alpha=0.7)
axes[0].set_ylabel('x detrended (arcsec)')
axes[0].set_title('Detrended Polar Motion X Component')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df.index, df['y_detrend'], 'r-', linewidth=0.5, alpha=0.7)
axes[1].set_ylabel('y detrended (arcsec)')
axes[1].set_title('Detrended Polar Motion Y Component')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/polar_motion_detrended.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Spectral Analysis (Identify Frequency Components)

Use FFT and Lomb-Scargle periodogram to identify the main frequency components:
- Annual: ~1 cycle/year = 0.00274 cycles/day
- Chandler: ~0.843 cycles/year = 0.00231 cycles/day (period ~433 days)

In [ ]:
# Use complex representation: m = x - i*y (prograde motion convention)
# This separates prograde and retrograde motions in the frequency domain

m_complex = df['x_detrend'].values - 1j * df['y_detrend'].values

# FFT
N = len(m_complex)
dt = 1.0  # 1 day sampling
freqs = fftfreq(N, dt)
fft_vals = fft(m_complex)

# Power spectrum (only positive frequencies for complex signal gives prograde)
power = np.abs(fft_vals)**2 / N

# Convert frequency to period in days
positive_mask = freqs > 0
periods = 1.0 / freqs[positive_mask]
power_positive = power[positive_mask]

In [ ]:
# Plot power spectrum with period on x-axis
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Full spectrum
ax1 = axes[0]
ax1.semilogy(periods, power_positive, 'b-', linewidth=0.5)
ax1.axvline(x=365.25, color='orange', linestyle='--', label='Annual (365.25 d)', alpha=0.8)
ax1.axvline(x=433, color='red', linestyle='--', label='Chandler (~433 d)', alpha=0.8)
ax1.axvline(x=182.6, color='green', linestyle='--', label='Semi-annual (182.6 d)', alpha=0.8)
ax1.set_xlabel('Period (days)')
ax1.set_ylabel('Power')
ax1.set_title('Power Spectrum of Complex Polar Motion (Prograde)')
ax1.set_xlim([10, 1000])
ax1.legend()
ax1.grid(True, alpha=0.3)

# Zoom on Chandler and Annual region
ax2 = axes[1]
mask = (periods > 200) & (periods < 600)
ax2.semilogy(periods[mask], power_positive[mask], 'b-', linewidth=1)
ax2.axvline(x=365.25, color='orange', linestyle='--', label='Annual (365.25 d)', alpha=0.8, linewidth=2)
ax2.axvline(x=433, color='red', linestyle='--', label='Chandler (~433 d)', alpha=0.8, linewidth=2)
ax2.set_xlabel('Period (days)')
ax2.set_ylabel('Power')
ax2.set_title('Power Spectrum - Zoom on Chandler/Annual Region')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/polar_motion_power_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Find the peak frequencies
# Look in the 300-500 day range for Chandler
chandler_mask = (periods > 350) & (periods < 500)
chandler_peak_idx = np.argmax(power_positive[chandler_mask])
chandler_period = periods[chandler_mask][chandler_peak_idx]

# Look in the 340-390 day range for Annual
annual_mask = (periods > 340) & (periods < 390)
annual_peak_idx = np.argmax(power_positive[annual_mask])
annual_period = periods[annual_mask][annual_peak_idx]

print("Detected Peaks:")
print("="*50)
print(f"Chandler wobble period: {chandler_period:.1f} days ({chandler_period/365.25:.3f} years)")
print(f"Annual wobble period: {annual_period:.1f} days")
print(f"\nTheoretical values:")
print(f"  Chandler: ~433 days (1.186 years)")
print(f"  Annual: 365.25 days")

## 5. Band-Pass Filter to Isolate Chandler Wobble

Design a Butterworth band-pass filter to isolate the Chandler wobble:
- Center frequency: 1/433 cycles/day
- Bandwidth: Allow ~380-500 day periods to capture the full Chandler band

In [ ]:
def bandpass_filter(data, lowcut_period, highcut_period, fs=1.0, order=4):
    """
    Apply Butterworth bandpass filter.
    
    Parameters:
    -----------
    data : array-like
        Input signal
    lowcut_period : float
        High-period cutoff (low frequency) in days
    highcut_period : float
        Low-period cutoff (high frequency) in days
    fs : float
        Sampling frequency (1/day for daily data)
    order : int
        Filter order
    
    Returns:
    --------
    Filtered signal
    """
    nyq = 0.5 * fs
    low = (1.0 / lowcut_period) / nyq  # Convert period to normalized frequency
    high = (1.0 / highcut_period) / nyq
    
    b, a = signal.butter(order, [low, high], btype='band')
    filtered = signal.filtfilt(b, a, data)
    return filtered

# Filter parameters for Chandler wobble (exclude annual)
# Chandler: ~433 days, Annual: ~365 days
# Use band from ~380 to ~520 days to capture Chandler while excluding annual
chandler_low = 520   # days (lower frequency limit)
chandler_high = 390  # days (upper frequency limit, between Chandler and Annual)

# Apply bandpass filter to detrended data
df['x_chandler'] = bandpass_filter(df['x_detrend'].values, chandler_low, chandler_high)
df['y_chandler'] = bandpass_filter(df['y_detrend'].values, chandler_low, chandler_high)

print(f"Chandler band-pass filter applied: {chandler_high}-{chandler_low} day periods")

In [ ]:
# Filter for Annual wobble as well (for comparison)
annual_low = 385   # days
annual_high = 345  # days

df['x_annual'] = bandpass_filter(df['x_detrend'].values, annual_low, annual_high)
df['y_annual'] = bandpass_filter(df['y_detrend'].values, annual_low, annual_high)

print(f"Annual band-pass filter applied: {annual_high}-{annual_low} day periods")

In [ ]:
# Plot isolated Chandler wobble
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Select a subset for clarity (e.g., 2000-2020)
plot_df = df.loc['2000':'2020']

axes[0].plot(plot_df.index, plot_df['x_chandler'], 'b-', linewidth=1, label='Chandler')
axes[0].plot(plot_df.index, plot_df['x_annual'], 'orange', linewidth=1, alpha=0.7, label='Annual')
axes[0].set_ylabel('x (arcsec)')
axes[0].set_title('Isolated Wobble Components - X')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(plot_df.index, plot_df['y_chandler'], 'r-', linewidth=1, label='Chandler')
axes[1].plot(plot_df.index, plot_df['y_annual'], 'orange', linewidth=1, alpha=0.7, label='Annual')
axes[1].set_ylabel('y (arcsec)')
axes[1].set_title('Isolated Wobble Components - Y')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/chandler_vs_annual_wobble.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Chandler Wobble Amplitude Analysis

The Chandler wobble amplitude varies over time, showing a notable minimum around 2005-2007.
We calculate the instantaneous amplitude using the envelope of the filtered signal.

In [ ]:
# Calculate Chandler wobble amplitude (radius in polar motion plane)
df['chandler_amplitude'] = np.sqrt(df['x_chandler']**2 + df['y_chandler']**2)

# Smooth the amplitude with a running mean (1 Chandler cycle ~ 433 days)
df['chandler_amplitude_smooth'] = df['chandler_amplitude'].rolling(window=433, center=True).mean()

# Calculate annual wobble amplitude for comparison
df['annual_amplitude'] = np.sqrt(df['x_annual']**2 + df['y_annual']**2)
df['annual_amplitude_smooth'] = df['annual_amplitude'].rolling(window=365, center=True).mean()

In [ ]:
# Plot amplitude time series
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df.index, df['chandler_amplitude_smooth'] * 1000, 'b-', linewidth=2, label='Chandler amplitude')
ax.plot(df.index, df['annual_amplitude_smooth'] * 1000, 'orange', linewidth=2, alpha=0.7, label='Annual amplitude')

ax.set_xlabel('Date')
ax.set_ylabel('Amplitude (mas)')
ax.set_title('Wobble Amplitudes Over Time (smoothed)')
ax.legend()
ax.grid(True, alpha=0.3)

# Mark notable events
ax.axvline(x=pd.Timestamp('2005-01-01'), color='red', linestyle=':', alpha=0.5)
ax.text(pd.Timestamp('2005-01-01'), ax.get_ylim()[1]*0.95, '2005\nminimum', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/wobble_amplitudes_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Polar plot of Chandler wobble trajectory
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Full history
ax1 = axes[0]
colors = np.linspace(0, 1, len(df))
scatter1 = ax1.scatter(df['x_chandler']*1000, df['y_chandler']*1000, c=colors, cmap='viridis', s=0.5, alpha=0.3)
ax1.set_xlabel('x_chandler (mas)')
ax1.set_ylabel('y_chandler (mas)')
ax1.set_title('Chandler Wobble - Full History')
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=ax1, label='Time')

# Recent 10 years (shows prograde circular motion)
recent = df.loc['2015':]
ax2 = axes[1]
colors2 = np.linspace(0, 1, len(recent))
scatter2 = ax2.scatter(recent['x_chandler']*1000, recent['y_chandler']*1000, c=colors2, cmap='plasma', s=2, alpha=0.5)
ax2.set_xlabel('x_chandler (mas)')
ax2.set_ylabel('y_chandler (mas)')
ax2.set_title('Chandler Wobble - 2015 to Present')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)
plt.colorbar(scatter2, ax=ax2, label='Time')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/chandler_wobble_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Phase Analysis

Analyze the phase of the Chandler wobble to look for phase jumps or anomalies.

In [ ]:
# Calculate instantaneous phase using Hilbert transform
from scipy.signal import hilbert

# Use complex representation for phase
chandler_complex = df['x_chandler'].values - 1j * df['y_chandler'].values

# Calculate phase (unwrapped)
phase = np.angle(chandler_complex)
phase_unwrapped = np.unwrap(phase)

# Convert to cycles (divide by 2*pi)
df['chandler_phase'] = phase_unwrapped / (2 * np.pi)

In [ ]:
# Plot phase evolution
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Cumulative phase (shows rotation rate)
ax1 = axes[0]
ax1.plot(df.index, df['chandler_phase'], 'b-', linewidth=1)
ax1.set_ylabel('Cumulative Phase (cycles)')
ax1.set_title('Chandler Wobble Phase Evolution')
ax1.grid(True, alpha=0.3)

# Expected linear phase (based on 433-day period)
expected_rate = 365.25 / 433  # cycles per year
years_elapsed = (df.index - df.index[0]).total_seconds() / (365.25 * 86400)
expected_phase = years_elapsed * expected_rate

# Phase residual (deviation from expected)
ax2 = axes[1]
# Align at start
phase_residual = df['chandler_phase'].values - expected_phase + expected_phase[0] - df['chandler_phase'].iloc[0]
ax2.plot(df.index, phase_residual, 'r-', linewidth=1)
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('Phase Residual (cycles)')
ax2.set_xlabel('Date')
ax2.set_title('Chandler Phase Residual (observed - expected at 433d period)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/chandler_phase_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary Statistics

In [ ]:
# Create summary table
summary_data = {
    'Metric': [
        'Data Start',
        'Data End', 
        'Total Days',
        'Detected Chandler Period (days)',
        'Mean Chandler Amplitude (mas)',
        'Min Chandler Amplitude (mas)',
        'Max Chandler Amplitude (mas)',
        'Mean Annual Amplitude (mas)',
        'Polar Drift Rate (mas/yr)',
        'Polar Drift Direction'
    ],
    'Value': [
        str(df.index.min().date()),
        str(df.index.max().date()),
        len(df),
        f"{chandler_period:.1f}",
        f"{df['chandler_amplitude_smooth'].mean()*1000:.1f}",
        f"{df['chandler_amplitude_smooth'].min()*1000:.1f}",
        f"{df['chandler_amplitude_smooth'].max()*1000:.1f}",
        f"{df['annual_amplitude_smooth'].mean()*1000:.1f}",
        f"{np.sqrt(x_rate_mas_yr**2 + y_rate_mas_yr**2):.1f}",
        f"{np.degrees(np.arctan2(y_rate_mas_yr, x_rate_mas_yr)):.1f}° from x-axis"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\nSummary Statistics")
print("="*60)
print(summary_df.to_string(index=False))

# Save to CSV for GitHub artifact
summary_df.to_csv(f'{FIGURES_DIR}/chandler_wobble_summary.csv', index=False)
print(f"\nSaved: {FIGURES_DIR}/chandler_wobble_summary.csv")

## 9. GitHub Artifacts Generation Guide

To generate artifacts suitable for GitHub:

### Figures
All figures are automatically saved to the `figures/` directory in PNG format at 150 DPI.

### Summary Tables
- CSV format for data tables (saved above)
- Can also export to Markdown format

### Including in GitHub README/Issues/PRs
Use relative paths to include figures:
```markdown
![Chandler Wobble Analysis](figures/chandler_wobble_trajectory.png)
```

In [ ]:
# Export summary as Markdown table
markdown_table = summary_df.to_markdown(index=False)
print("\nMarkdown Table (copy-paste for GitHub):")
print("="*60)
print(markdown_table)

# Save markdown summary
with open(f'{FIGURES_DIR}/chandler_wobble_summary.md', 'w') as f:
    f.write("# Chandler Wobble Analysis Summary\n\n")
    f.write(markdown_table)
    f.write("\n\n## Figures\n\n")
    f.write("- `polar_motion_raw_timeseries.png` - Raw polar motion x,y components\n")
    f.write("- `polar_motion_trajectory.png` - Polar motion in x-y plane showing drift\n")
    f.write("- `polar_motion_detrended.png` - Detrended polar motion\n")
    f.write("- `polar_motion_power_spectrum.png` - FFT power spectrum\n")
    f.write("- `chandler_vs_annual_wobble.png` - Filtered Chandler vs Annual components\n")
    f.write("- `wobble_amplitudes_timeseries.png` - Amplitude variations over time\n")
    f.write("- `chandler_wobble_trajectory.png` - Isolated Chandler motion\n")
    f.write("- `chandler_phase_analysis.png` - Phase evolution and residuals\n")

print(f"\nSaved: {FIGURES_DIR}/chandler_wobble_summary.md")

In [ ]:
# List all generated artifacts
import os
print("\nGenerated Artifacts:")
print("="*60)
for f in sorted(os.listdir(FIGURES_DIR)):
    filepath = os.path.join(FIGURES_DIR, f)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  {f:50s} {size_kb:8.1f} KB")

## 10. Next Steps and Advanced Analysis

### Further Analysis Options:

1. **Wavelet Analysis**: Time-frequency decomposition to see how Chandler amplitude/frequency evolve
2. **Excitation Function Analysis**: Compare with atmospheric (AAM), oceanic (OAM), hydrological angular momentum
3. **Cross-correlation with geophysical events**: Earthquakes, volcanic eruptions, etc.
4. **Q-factor estimation**: Measure the damping of the Chandler wobble
5. **Prediction/Forecasting**: Use the extracted signal for short-term polar motion prediction

### Key References:
- Gross, R.S. (2000). "The excitation of the Chandler wobble." Geophysical Research Letters.
- Chao, B.F. (1985). "On the excitation of the Earth's polar motion." Geophysical Research Letters.
- Vondrák, J. (1999). "Earth rotation parameters 1899.7–1992.0." Surveys in Geophysics.

In [ ]:
print("\nAnalysis Complete!")
print("="*60)
print(f"Figures saved to: {os.path.abspath(FIGURES_DIR)}")
print("\nTo include in GitHub markdown:")
print('![Figure](figures/chandler_wobble_trajectory.png)')